<a href="https://colab.research.google.com/github/juanes0789/Pruebas-calidad/blob/main/Pruebas_Software_Caja_Negra_y_Blanca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Técnicas de Prueba de Software: Partición de Equivalencia y Cobertura de Caminos

Integrantes:

>Mosquera Perea Juan Esteban,
Mosquera Alvarez Maria Paula,
Reyes Uribe Mateo


---

## 2. Objetivo
El objetivo de este notebook es demostrar en la práctica cómo diseñar y ejecutar casos de prueba utilizando dos técnicas fundamentales de calidad de software:
*   **Caja Negra (Partición de Equivalencia):** Centrada en el dominio de las variables de entrada sin considerar la estructura de código interna. Divide el dominio en clases y selecciona representantes válidos e inválidos.
*   **Caja Blanca (Cobertura de Caminos):** Centrada en el análisis del flujo interno de control (decisiones y ramificaciones) del programa para asegurar el recorrido de rutas específicas.
*   **pytest:** Utilizaremos este framework como herramienta para automatizar de manera ágil e inequívoca la ejecución y validación mediante aserciones (`assert`).

## 3. Contexto del sistema bajo prueba
Para esta demostración utilizaremos una función denominada `clasificar_hogar()` que simula un modelo de clasificación de hogares según variables de entrada:

*   **Ingreso**
*   **Vivienda**
*   **Tiene Vehículo**
*   **Educación**

Dependiendo del puntaje obtenido, se asigna una categoría (Grupo A, B, C o D).



In [ ]:
%%writefile sistema.py
# Código del sistema bajo prueba

def clasificar_hogar(ingreso, vivienda, tiene_vehiculo, educacion):
    # Validaciones básicas
    if ingreso < 0:
        return {"error": "El ingreso no puede ser negativo"}
    if vivienda not in ["Propia", "Arrendada", "Familiar"]:
        return {"error": "Tipo de vivienda inválido"}
    if educacion not in ["Primaria", "Secundaria", "Universitaria"]:
        return {"error": "Nivel de educación inválido"}

    puntaje = 0

    # Evaluar ingreso
    if ingreso < 2000000:
        puntaje -= 20
    elif ingreso <= 5000000:
        puntaje += 10
    else:
        puntaje += 30

    # Evaluar vivienda
    if vivienda == "Propia":
        puntaje += 20
    elif vivienda == "Arrendada":
        puntaje += 5
    elif vivienda == "Familiar":
        puntaje += 0

    # Evaluar vehículo
    if tiene_vehiculo:
        puntaje += 15

    # Evaluar educación
    if educacion == "Primaria":
        puntaje += 0
    elif educacion == "Secundaria":
        puntaje += 5
    elif educacion == "Universitaria":
        puntaje += 15

    # Clasificación final
    if puntaje < 20:
        grupo = "A"
    elif puntaje < 50:
        grupo = "B"
    elif puntaje < 80:
        grupo = "C"
    else:
        grupo = "D"

    return {
        "puntaje": puntaje,
        "grupo": grupo
    }

Writing sistema.py


# --- CAJA NEGRA: PARTICIÓN DE EQUIVALENCIA ---

## 4. ¿Qué es partición de equivalencia?
La **partición de equivalencia** es una técnica de *caja negra*. No analizamos la implementación interna del código; dividimos el dominio de entrada (todos los valores posibles) en clases equivalentes.

Asumimos que el sistema tratará de la misma forma a todos los valores dentro de una misma clase, por lo que seleccionamos un **valor representante** por cada clase.

## 5. Clases de Equivalencia Diseñadas

### Ingreso:

| Clase | Condición | Tipo | Representante |
| :--- | :--- | :--- | :--- |
| **CE1** | `ingreso < 0` | Inválida | `-100000` |
| **CE2** | `0 ≤ ingreso < 2000000` | Válida | `1500000` |
| **CE3** | `2000000 ≤ ingreso ≤ 5000000` | Válida | `3000000` |
| **CE4** | `ingreso > 5000000` | Válida | `8000000` |

### Otras Variables de Entrada:

| Variable | Clase de Equivalencia (Válida) | Clase de Equivalencia (Inválida) |
| :--- | :--- | :--- |
| **Vivienda** | `"Propia"`, `"Arrendada"`, `"Familiar"` | Cualquier otro valor (Ej. `"Hotel"`) |
| **Educación** | `"Primaria"`, `"Secundaria"`, `"Universitaria"` | Cualquier otro valor (Ej. `"Doctorado"`) |
| **Vehículo** | `True`, `False` | No aplica |

## 6. Diseño de Casos de Prueba (Caja Negra)

Crearemos un archivo con pruebas para validar de forma estructurada con `pytest` las clases de equivalencia principales.

In [ ]:
# Instalar y configurar pytest para su ejecución en entorno Colab
!pip install pytest -q
import pytest

In [ ]:
%%writefile test_caja_negra.py
# Archivo de pruebas de caja negra para pytest
from sistema import clasificar_hogar

def test_ce1_ingreso_negativo():
    resultado = clasificar_hogar(-100000, "Propia", True, "Universitaria")
    assert "error" in resultado
    assert resultado["error"] == "El ingreso no puede ser negativo"

def test_ce2_ingreso_bajo():
    # Ingreso: 1.500.000 (-20), Vivienda: Familiar (0), Vehículo: False (0), Educación: Primaria (0) => Puntaje: -20, Grupo A
    resultado = clasificar_hogar(1500000, "Familiar", False, "Primaria")
    assert resultado == {"puntaje": -20, "grupo": "A"}

def test_ce3_ingreso_medio():
    # Ingreso: 3.000.000 (10), Vivienda: Arrendada (5), Vehículo: False (0), Educación: Secundaria (5) => Puntaje: 20, Grupo B
    resultado = clasificar_hogar(3000000, "Arrendada", False, "Secundaria")
    assert resultado == {"puntaje": 20, "grupo": "B"}

def test_ce4_ingreso_alto():
    # Ingreso: 8.000.000 (30), Vivienda: Propia (20), Vehículo: True (15), Educación: Universitaria (15) => Puntaje: 80, Grupo D
    resultado = clasificar_hogar(8000000, "Propia", True, "Universitaria")
    assert resultado == {"puntaje": 80, "grupo": "D"}

def test_vivienda_invalida():
    resultado = clasificar_hogar(3000000, "Hotel", True, "Universitaria")
    assert "error" in resultado
    assert resultado["error"] == "Tipo de vivienda inválido"

def test_educacion_invalida():
    resultado = clasificar_hogar(3000000, "Propia", True, "Doctorado")
    assert "error" in resultado
    assert resultado["error"] == "Nivel de educación inválido"

Overwriting test_caja_negra.py


In [ ]:
# Ejecutar pruebas de Caja Negra con pytest
!pytest test_caja_negra.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, typeguard-4.6.0, anyio-4.14.2
collected 6 items                                                              

test_caja_negra.py::test_ce1_ingreso_negativo PASSED                     [ 16%]
test_caja_negra.py::test_ce2_ingreso_bajo PASSED                         [ 33%]
test_caja_negra.py::test_ce3_ingreso_medio PASSED                        [ 50%]
test_caja_negra.py::test_ce4_ingreso_alto PASSED                         [ 66%]
test_caja_negra.py::test_vivienda_invalida PASSED                        [ 83%]
test_caja_negra.py::test_educacion_invalida PASSED                       [100%]

============================== 6 passed in 0.02s ===============================


# --- CAJA BLANCA: COBERTURA DE CAMINOS ---

## 7. ¿Qué es cobertura de caminos?
La **cobertura de caminos** es una técnica de *caja blanca* en la que conocemos la implementación interna. Analizamos las decisiones y ramificaciones de control del programa para estructurar casos de prueba que viajen por rutas específicas.

### Decisiones identificadas en el código:
1. ¿Ingreso negativo?
2. ¿Vivienda válida?
3. ¿Educación válida?
4. Rangos de Ingreso (Bajo, Medio, Alto)
5. Tipo de Vivienda (Propia, Arrendada, Familiar)
6. ¿Tiene Vehículo?
7. Nivel de Educación (Primaria, Secundaria, Universitaria)
8. Rango final de Puntaje (Grupo A, B, C o D)

## 8. Grafo de flujo de control
Este diagrama representa con exactitud la lógica secuencial y condicional que acabamos de describir:

[![](https://mermaid.ink/img/pako:eNpdktt2ojAUhl8lK7eKi4OCsjqd5bm21dp66AG9yIJU0wHiiuB0BnyYeYBezSP4YhMTUByu-Pf359_sTRLoUg9DG7779Ke7RiwC084iBPxpOoOQuIQugaJcg1YyCFcMbym4Aur3vbS0jiidHL5S0NWcLmOUASJty6JjRFPQTuZkR3DoIbA7_PGJh_KU9snT1bOQXWZdFh2iTyfperGLXHL4G_4f1DkHGVkQPpmXRY-I6uUjZcd7ArXQBw_oO4quLov1IfYIBzdOSbsETT_i9YFTMvJ6X2zs9jRwln8jy1IMiuJWBI0Z3RCUgjundOotSZOxY47H4b1Tql2wHgqITxBLwdAp5cfuRPoomeP14cuNfZqv6F4CKYZFMTov5oHPmDcZnXY6Psc_iIOPhV-RxY8lkOIxm4oEiB3nejoHSDTBbszHEnBynkvCWUh2mG1JJPm08E1PosssGcdhhD5w1nsiq1JMi2ImAq90NQVzp8_iDQXNZRHpqlJtpOA5g60LWFMVi8OXDLYv4PW3Ok99zVgnY3PR_c3pkfzaPcuKFC9F8SoFLMMVIx60IxbjMgwwC9BRwuRoW8BojQO8gDZ_ZfxSfyp8bz8Ul_qULeAi3PPzGxS-URrkEYzGqzW035G_5SreeCjCHYJWDAWn6vFaYdamfJHQrlkNEQLtBH5C26hXK1rVqpv1hmaput6oleEvaCt6o1ExLVM1dU0zjJpZtfZl-Fs01iqqpdY1zayZlmEaVcPY_wO5ZUlT?type=png)](https://mermaid.live/edit#pako:eNpdkltymzAUhrei0atNBgMGw6Tp-Jo4F8eJL7lgP2hAsZUC8ijgpgUvpgvIU5fgjVVIYOPyxH--X__hHJRCj_oYOvAtoD-9NWIxmPYWEeBP2x1GxCN0CRTlAnTSYbRi-IOCc6B-30lLJ0fZZP-VgX7D7TNGGSDStqw6RjQD3XROtgRHPgLb_Z-A-KhM6R48fa0I2RbWZdUh-vTSvp94yCP7v9H_Qb1jkF4E4YN5WfWIqEE5UnF8IFAHvfOAS1fR1GW1fod9wsGVW2ucgnYQ8_rQrell_VJs7PowcJF_JctSDKviWgSNGd0QlIEbt3boLUmbsTzH5_DWrTVP2ACFJCCIZeDOrZXHbkT6KJ3j9f7LSwJaruhWAinuqmJ0XMw9n7FsMjrsdHyMvxcHHyq_oogfSyDFQzEVCRHL53o8Bkg0wV7CxxJwcpxLwllEtph9kFjyaeWbHkWXWTpOohi946L3RFalmFbFTASea2oG5u4lSzYUtJdVpKmKYWfgqYCdE9hUFYvD5wJ2T-DFtxZPfSlYr2Bz0f3VHZDy2j3JihTPVfEiBazDFSM-dGKW4DoMMQtRLmGa2xYwXuMQL6DDXxm_1J8K39sPxaMBZQu4iHb8_AZFr5SGZQSjyWoNnTcUfHCVbHwU4x5BK4bCQzW_Vph1KV8kdJq6LUKgk8JP6CgN3TzTdctumJbdMvWWYdXhr7xuG80zu2XblmY01Kahmbs6_C06G2dayzYt1TTsZquhWebuH7iSSW8)

## 9. Identificación de caminos representativos
Diseñamos casos específicos para forzar la ejecución a lo largo de rutas representativas en el código.

*Nota: Estos tres casos no agotan la totalidad de combinaciones matemáticas, sino que ejemplifican de forma didáctica la cobertura de caminos representativos.*

| Camino | Ingreso | Vivienda | Vehículo | Educación | Ruta Lógica Recorrida |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **CP_C1** | 1,500,000 | Familiar | No | Primaria | Ingreso bajo → vivienda familiar → sin vehículo → primaria → clasificación final Grupo A |
| **CP_C2** | 3,000,000 | Arrendada | No | Secundaria | Ingreso medio → vivienda arrendada → sin vehículo → secundaria → clasificación final Grupo B |
| **CP_C3** | 8,000,000 | Propia | Sí | Universitaria | Ingreso alto → vivienda propia → con vehículo → universitaria → clasificación final Grupo D |

## 10. Creación de pruebas de cobertura de caminos con pytest

In [ ]:
%%writefile test_caja_blanca.py
# Archivo de pruebas de caja blanca para pytest
from sistema import clasificar_hogar

def test_camino_1():
    # CP_C1: Ingreso bajo, Familiar, Sin Vehículo, Primaria
    resultado = clasificar_hogar(1500000, "Familiar", False, "Primaria")
    assert resultado == {
        "puntaje": -20,
        "grupo": "A"
    }

def test_camino_2():
    # CP_C2: Ingreso medio, Arrendada, Sin Vehículo, Secundaria
    resultado = clasificar_hogar(3000000, "Arrendada", False, "Secundaria")
    assert resultado == {
        "puntaje": 20,
        "grupo": "B"
    }

def test_camino_3():
    # CP_C3: Ingreso alto, Propia, Con Vehículo, Universitaria
    resultado = clasificar_hogar(8000000, "Propia", True, "Universitaria")
    assert resultado == {
        "puntaje": 80,
        "grupo": "D"
    }


Overwriting test_caja_blanca.py


In [ ]:
# Ejecutar pruebas de Caja Blanca con pytest
!pytest test_caja_blanca.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, typeguard-4.6.0, anyio-4.14.2
collected 3 items                                                              

test_caja_blanca.py::test_camino_1 PASSED                                [ 33%]
test_caja_blanca.py::test_camino_2 PASSED                                [ 66%]
test_caja_blanca.py::test_camino_3 PASSED                                [100%]

============================== 3 passed in 0.01s ===============================




## 11. Diferencia entre las dos técnicas

| Aspecto | Partición de equivalencia | Cobertura de caminos |
| :--- | :--- | :--- |
| **Tipo de Prueba** | Caja negra | Caja blanca |
| **¿Se conoce el código?** | No es necesario | Sí, es indispensable |
| **¿Qué se analiza?** | Especificación, entradas y clases representantes | Estructura, decisiones y flujo de control interno |
| **Objetivo Principal** | Reducir el número de casos de prueba cubriendo el dominio | Asegurar la ejecución de las diferentes rutas lógicas |
| **Ejemplo de Caso** | Probar `ingreso = 3000000` (como representante de clase) | Ruta: Bajo → Familiar → Sin vehículo → Primaria |
| **Herramienta de Automatización** | `pytest` | `pytest` |

---

## 12. Conclusiones

1. **La partición de equivalencia** permite optimizar los recursos reduciendo drásticamente la cantidad de pruebas necesarias mediante representantes representativos del dominio.
2. **La cobertura de caminos** de caja blanca nos obliga a entender a nivel de código cada bifurcación lógica, previniendo caminos muertos o ramas no deseadas.
3. Ambas técnicas son complementarias; el uso del framework **pytest** facilita automatizar las aserciones de forma repetitiva y confiable tanto para caja negra como caja blanca.